# LCProp PR

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcroning/LCProp/blob/v0.5.0rc3/notebooks/releases/0.5.0rc3/lcprop_pr_colab.ipynb)

Start with a fresh Colab CPU runtime and the reviewed LCProp wheel with its Colab extra. Verify the wheel checksum and restart after installation before running the display cell. Only the interface Run button starts propagation.

This rc3 notebook uses CPU for both reduced PR workflows.


## Install once in a fresh CPU runtime

For candidate review, upload the pinned wheel to Colab session storage. After publication, this cell downloads the exact wheel automatically when it is absent. It checks SHA-256 before installing. Restart the session, then run Verify installation.


In [ ]:
import hashlib
from pathlib import Path
import subprocess
import sys
from urllib.request import urlopen

wheel = Path("/content/lcprop-0.5.0rc3-py3-none-any.whl")
expected_sha256 = "44840b22a3fc8d8b49059bbc5baa1d63bef6c156f4c0d982f7df7ab356e55f94"
url = "https://github.com/mcroning/LCProp/releases/download/v0.5.0rc3/" + wheel.name
if wheel.is_file():
    wheel_bytes = wheel.read_bytes()
else:
    with urlopen(url, timeout=60) as response:
        wheel_bytes = response.read()
if hashlib.sha256(wheel_bytes).hexdigest() != expected_sha256:
    raise ValueError("The wheel does not match the pinned SHA-256.")
if not wheel.is_file():
    wheel.write_bytes(wheel_bytes)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps", str(wheel)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "lcprop[colab]==0.5.0rc3", "jedi"],
    check=True,
)
print("Restart the session and continue at Verify installation.")

## Verify installation

Run this cell after restarting. Then run the interface cell below.

In [ ]:
from importlib import metadata
import json
import subprocess
import sys

expected_sha256 = "44840b22a3fc8d8b49059bbc5baa1d63bef6c156f4c0d982f7df7ab356e55f94"
dist = metadata.distribution("lcprop")
origin = json.loads(dist.read_text("direct_url.json") or "{}")
installed_sha256 = origin.get("archive_info", {}).get("hashes", {}).get("sha256")
if dist.version != "0.5.0rc3" or installed_sha256 != expected_sha256:
    raise RuntimeError("Install the pinned wheel and restart this session first.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
print("Verified LCProp", dist.version, installed_sha256)

In [ ]:
from google.colab import output
from IPython import get_ipython
from IPython.display import clear_output
from lcprop.notebook import PRNotebook

if "lcprop_pr" not in globals():
    output.enable_custom_widget_manager()
    get_ipython().run_line_magic("matplotlib", "widget")
    lcprop_pr = PRNotebook()
elif lcprop_pr.session.disposed:
    raise RuntimeError("Restart after saving needed files to replace a disposed controller.")
clear_output(wait=True)
lcprop_pr.display_in_colab()
